# Multi-seed validation of the DAN, DANN, and DAN-DG failures

This supplementary notebook leaves the Task 2 and Task 3 notebooks unchanged. It retrains the suspicious configurations with two additional seeds (`6305`, `6306`) and evaluates their selected checkpoints on PACS Sketch.

The original seed-6304 train/validation split is held fixed. Only model initialization, minibatch order, stochastic augmentation, and DANN dropout vary. This isolates training-seed sensitivity from split sensitivity. Target labels are used only after source-validation checkpoint selection.

Pre-registered diagnostics: a run is called collapsed when mean source accuracy is below 0.30 **and** Sketch accuracy is below 0.10. A recorded loss above 1,000 is flagged as extreme.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import torch
from IPython.display import Image, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'validation':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation.collapse_multiseed import (
    OUT, VALIDATION_SEEDS, check_protocol_only, protocol_record, run_validation
)

print('CUDA available:', torch.cuda.is_available())
print(json.dumps(protocol_record(), indent=2))

## 1. Data and split check

This cell reads the fixed source split and verifies the original PACS domain sizes. It performs no training and does not inspect target labels for model selection.

In [ ]:
protocol_check = check_protocol_only()
pd.DataFrame({
    'source_train': protocol_check['source_train_sizes'],
    'source_validation': protocol_check['source_validation_sizes'],
})

## 2. Additional-seed training and evaluation

A CUDA GPU is required by default. The helper saves each best checkpoint and curve immediately, so an interrupted run can resume without repeating completed configurations. Pass `force=True` only to deliberately replace validation checkpoints.

In [ ]:
combined_results, aggregate_summary = run_validation(force=False)

## 3. Results

The combined table includes the original seed 6304 for context. The aggregate table reports collapse and extreme-loss counts across all three seeds.

In [ ]:
display(combined_results)
display(aggregate_summary)
display(Image(filename=str(OUT / 'multiseed_accuracy.png')))

## Interpretation checklist

After the runs finish, determine from the generated tables—not from a single checkpoint—whether:

- DAN and DAN-DG meet the pre-registered collapse definition for both new seeds;
- DANN repeatedly exhibits extreme loss even if its accuracy does not meet the collapse definition;
- the result is seed-stable or driven by the original run; and
- the conclusion should be described as consistent over three seeds rather than universally guaranteed.

Write the report interpretation independently in accordance with the assignment's AI policy.